In [ ]:
import pandas as pd

df_marking = pd.read_csv('df_with_marking_final.csv')

In [ ]:
import re

def get_full_text(row):
    return f"{str(row['preamble'])} {str(row['description'])} {str(row['sentence'])}"

female_forms = ["потерпевшая", "потерпевшей", "потерпевшую"]
female_pattern = r"\b(" + "|".join(female_forms) + r")\b"

train_data_has_woman_victim = []

for idx, row in df_marking.iterrows():
    text = get_full_text(row)
    text_lower = text.lower()

    has_woman = row["has_woman_victim"]
    if pd.isnull(has_woman):
        continue

    if has_woman == 1:
        match = re.search(female_pattern, text_lower)
        if match:
            start, end = match.span()
            train_data_has_woman_victim.append((text, {"entities": [(start, end, "HAS_WOMAN_VICTIM")]}))
        else:
            print(f"Не найдена форма 'потерпевшая' при has_woman_victim = 1 в id={row['id']}")
    else:
        # В случае 0 - пустая разметка
        train_data_has_woman_victim.append((text, {"entities": []}))
        pass

print(f"TRAIN_DATA_HAS_WOMAN_VICTIM готово: {len(train_data_has_woman_victim)} примеров")

TRAIN_DATA_HAS_WOMAN_VICTIM готово: 100 примеров


In [ ]:
import spacy
from spacy.training import Example
from spacy.util import minibatch
import random
import warnings

nlp = spacy.blank("ru")

if "ner" not in nlp.pipe_names:
    ner = nlp.add_pipe("ner")
else:
    ner = nlp.get_pipe("ner")

ner.add_label("HAS_WOMAN_VICTIM")

examples = []
for text, annot in train_data_has_woman_victim:
    doc = nlp.make_doc(text)
    example = Example.from_dict(doc, annot)
    examples.append(example)

optimizer = nlp.begin_training()
warnings.filterwarnings("ignore")

for i in range(15):
    random.shuffle(examples)
    losses = {}
    batches = minibatch(examples, size=8)
    for batch in batches:
        nlp.update(batch, drop=0.2, losses=losses)
    print(f"Epoch {i + 1}, Losses: {losses}")

nlp.to_disk("ner_has_woman_victim_model")
print("Модель сохранена в 'ner_has_woman_victim_model'")

Epoch 1, Losses: {'ner': 260701.56233640385}
Epoch 2, Losses: {'ner': 53.99853678644714}
Epoch 3, Losses: {'ner': 52.46302553411195}
Epoch 4, Losses: {'ner': 1565.754295175186}
Epoch 5, Losses: {'ner': 97.26339550164934}
Epoch 6, Losses: {'ner': 38.430907440266324}
Epoch 7, Losses: {'ner': 43.76460353260944}
Epoch 8, Losses: {'ner': 39.34779050378038}
Epoch 9, Losses: {'ner': 35.3969744061667}
Epoch 10, Losses: {'ner': 23.37543289816519}
Epoch 11, Losses: {'ner': 38.52843585671119}
Epoch 12, Losses: {'ner': 22.759725743788287}
Epoch 13, Losses: {'ner': 19.709812678486717}
Epoch 14, Losses: {'ner': 17.678152853039506}
Epoch 15, Losses: {'ner': 21.78183129006086}
✅ Модель сохранена в 'ner_has_woman_victim_model'


In [ ]:
from sklearn.metrics import accuracy_score, f1_score

nlp = spacy.load("ner_has_woman_victim_model")

y_true = []
y_pred = []

def get_full_text(row):
    return f"{str(row['preamble'])} {str(row['description'])} {str(row['sentence'])}"

for _, row in df_marking.iterrows():
    true_val = row["has_woman_victim"]
    if pd.isnull(true_val):
        continue

    text = get_full_text(row)
    doc = nlp(text.lower())

    predicted_val = 0
    for ent in doc.ents:
        if ent.label_ == "HAS_WOMAN_VICTIM":
            print(ent.text)
            predicted_val = 1
            break

    y_true.append(int(true_val))
    y_pred.append(predicted_val)

accuracy = accuracy_score(y_true, y_pred)
f1 = f1_score(y_true, y_pred, average='weighted')

print(f"Accuracy: {accuracy:.2%}")
print(f"F1-score: {f1:.2%}")

потерпевшая
потерпевшей
потерпевшей
потерпевшую
потерпевшей
потерпевшей
потерпевшей
Accuracy: 80.00%
F1-score: 75.32%
